<div style="background:linear-gradient(135deg,#001F3F 0%,#0093D5 100%);padding:40px 32px;border-left:6px solid #EE3A43;">
<h1 style="color:#fff;font-family:Arial,sans-serif;font-size:28px;margin:0 0 8px;">02 — KPI Forecasting</h1>
<h2 style="color:rgba(255,255,255,.75);font-family:Arial,sans-serif;font-size:16px;font-weight:400;margin:0 0 20px;">SpiriCom · NOC Intelligence Platform · Huawei Technologies Tunisia</h2>
<div style="display:flex;gap:32px;flex-wrap:wrap;">
<div style="color:rgba(255,255,255,.65);font-size:13px;"><strong style="color:#fff;">Input</strong><br/>data/processed/kpi_clean.parquet</div>
<div style="color:rgba(255,255,255,.65);font-size:13px;"><strong style="color:#fff;">Models</strong><br/>Prophet · ARIMA (pmdarima) · XGBoost</div>
<div style="color:rgba(255,255,255,.65);font-size:13px;"><strong style="color:#fff;">Targets</strong><br/>session_flag · traffic_5G · brand traffic</div>
<div style="color:rgba(255,255,255,.65);font-size:13px;"><strong style="color:#fff;">Outputs</strong><br/>models/prediction/ + forecast_results.json</div>
</div></div>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 0.  IMPORTS, STYLE & PATHS
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy  as np
import json, warnings, os
from pathlib import Path
from datetime import datetime

# ── Visualization ──────────────────────────────────────────────────────
import matplotlib.pyplot   as plt
import matplotlib.patches  as mpatches
import matplotlib.colors   as mcolors
import seaborn as sns
import plotly.express       as px
import plotly.graph_objects as go
import plotly.io            as pio

# ── ML ────────────────────────────────────────────────────────────────
from sklearn.model_selection  import train_test_split, TimeSeriesSplit
from sklearn.preprocessing    import LabelEncoder
from sklearn.metrics          import (
    f1_score, precision_score, recall_score, roc_auc_score,
    accuracy_score, mean_absolute_error, mean_squared_error, r2_score,
    confusion_matrix, classification_report
)
import xgboost as xgb

# ── Time-series ───────────────────────────────                                                                                                                        ─────────────────────────
from prophet import Prophet
try:
    from pmdarima import auto_arima
    PMDARIMA_OK = True
except ImportError:
    PMDARIMA_OK = False
    print('⚠️  pmdarima not installed — ARIMA section will be skipped')
    print('   pip install pmdarima')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────
PROC_DIR   = Path('data/processed')
FIG_DIR    = Path('data/outputs/figures')
OUT_DIR    = Path('data/outputs')
MODEL_DIR  = Path('models/prediction')
for d in [FIG_DIR, OUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Huawei Brand Palette ───────────────────────────────────────────────
HW = dict(
    blue='#0093D5', red='#EE3A43', navy='#001F3F', cyan='#00C3FF',
    green='#22C55E', amber='#F59E0B', purple='#8B5CF6', orange='#F97316',
    muted='#6B7280',
)
PALETTE = [HW['blue'], HW['red'], HW['cyan'], HW['green'],
           HW['amber'], HW['purple'], HW['orange'], HW['navy']]

# ── Matplotlib rcParams ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.edgecolor':'#E5E7EB', 'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'axes.grid.axis':'y', 'grid.color':'#F3F4F6', 'grid.linewidth':0.8,
    'axes.labelcolor':HW['navy'], 'axes.labelweight':'bold', 'axes.labelsize':11,
    'axes.titlesize':13, 'axes.titleweight':'bold', 'axes.titlecolor':HW['navy'],
    'xtick.color':HW['muted'], 'ytick.color':HW['muted'],
    'xtick.labelsize':9, 'ytick.labelsize':9,
    'legend.fontsize':9, 'legend.framealpha':0.92, 'legend.edgecolor':'#E5E7EB',
    'font.family':'DejaVu Sans', 'figure.dpi':120,
    'savefig.dpi':300, 'savefig.bbox':'tight', 'savefig.facecolor':'white',
})

# ── Plotly template ────────────────────────────────────────────────────
pio.templates.default = 'plotly_white'
PLOTLY_LAYOUT = dict(
    font=dict(family='Arial, sans-serif', color=HW['navy']),
    paper_bgcolor='white', plot_bgcolor='white',
    title_font=dict(size=16, color=HW['navy']),
    colorway=PALETTE,
    margin=dict(l=60, r=40, t=80, b=60),
    legend=dict(bgcolor='rgba(255,255,255,.9)', bordercolor='#E5E7EB', borderwidth=1),
)

# ── Helpers ────────────────────────────────────────────────────────────
def add_watermark(fig, text='SpiriCom · Huawei Technologies Tunisia'):
    fig.text(0.99, 0.01, text, ha='right', va='bottom',
             fontsize=7, color=HW['muted'], style='italic',
             transform=fig.transFigure)

def save_fig(name):
    path = FIG_DIR / f'{name}.png'
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f'  Saved: {path}')

def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

print(' Setup complete')
print(f'   Model outputs: {MODEL_DIR.resolve()}')
print(f'   Figures:       {FIG_DIR.resolve()}')

---
##  Section 1 — Load Data & Feature Audit

In [ ]:
# ── 1.1  Load kpi_clean.parquet ────────────────────────────────────────
df = pd.read_parquet(PROC_DIR / 'kpi_clean.parquet')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Identify key columns (post NB00 drop of >=80% null columns)
has_5g       = 'traffic_5g' in df.columns or 'traffic_5g' in [c.lower() for c in df.columns]
has_session  = 'session_flag' in df.columns
has_brand    = 'brand' in df.columns
has_ts       = 'timestamp' in df.columns

# Normalise column names to lowercase for safety
df.columns = df.columns.str.lower()

print(f'  Has traffic_5g   : {has_5g}')
print(f'  Has session_flag : {has_session}')
print(f'  Has brand        : {has_brand}')
print(f'  Has timestamp    : {has_ts}')
df.head(0)

In [ ]:
# ── 1.2  Column audit ─────────────────────────────────────────────────
print('=== ALL COLUMNS AFTER NB00 PREPROCESSING ===')
null_pct = df.isnull().mean() * 100
audit = pd.DataFrame({
    'dtype'    : df.dtypes,
    'null_pct' : null_pct.round(2),
    'n_unique' : df.nunique(),
}).sort_values('null_pct', ascending=False)
print(audit.to_string())
print(f'\nMax null % in any retained column: {null_pct.max():.1f}%  (should be < 80%)')

In [ ]:
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'], dayfirst=True, errors='coerce')

    print("Missing timestamps:", df['timestamp'].isna().sum())

    df = df.dropna(subset=['timestamp']) \
           .sort_values('timestamp') \
           .reset_index(drop=True)

    print(f'Timestamp range: {df["timestamp"].min()} → {df["timestamp"].max()}')
    print(f'Rows after timestamp filter: {len(df):,}')

else:
    df['timestamp'] = pd.date_range(start='2025-01-01', periods=len(df), freq='H')

# Features time-series
df['ds'] = df['timestamp']
df['date'] = df['timestamp'].dt.date
df['date_str'] = df['timestamp'].dt.strftime('%d/%m/%Y')

---
## 📊 Section 2 — Target Engineering

In [ ]:
# ── 2.1  session_flag — binary target ────────────────────────────────
if 'session_flag' in df.columns:
    print('session_flag distribution:')
    print(df['session_flag'].value_counts())
    print(f'Active rate: {df["session_flag"].mean():.3f}')

    # ── VERIFY: active users should have higher traffic ───────────────
    # If group 0 has higher mean traffic than group 1, the flag is inverted
    verify_cols = [c for c in ['dou_total','traffic_4g','traffic_5g','duration']
                   if c in df.columns]
    if verify_cols:
        print('\nMean traffic/duration by session_flag value:')
        print(df.groupby('session_flag')[verify_cols].mean().round(2))

    # ── AUTO-CORRECT if inverted ──────────────────────────────────────
    # Rule: if flag=0 group has higher mean dou_total, the label is inverted
    dou_col = next((c for c in ['dou_total','traffic_4g'] if c in df.columns), None)
    if dou_col:
        mean_0 = df.loc[df['session_flag']==0, dou_col].mean()
        mean_1 = df.loc[df['session_flag']==1, dou_col].mean()

        if mean_0 > mean_1:
            print(f'\n⚠️  Flag appears INVERTED (group 0 has higher {dou_col}: {mean_0:.2f} vs {mean_1:.2f})')
            print('   Inverting session_flag so that 1 = active session...')
            df['session_flag'] = 1 - df['session_flag']
            print(f'   New distribution:\n{df["session_flag"].value_counts()}')
            print(f'   New active rate: {df["session_flag"].mean():.3f}')
        else:
            print(f'\n✅  Flag direction confirmed (group 1 has higher {dou_col}: {mean_1:.2f} vs {mean_0:.2f})')

    SESSION_TARGET = 'session_flag'

else:
    dur_col = next((c for c in df.columns if 'duration' in c), None)
    if dur_col:
        df['session_flag'] = (df[dur_col] > 0).astype(int)
        print(f'session_flag derived from {dur_col}')
        SESSION_TARGET = 'session_flag'
    else:
        SESSION_TARGET = None
        print('⚠️  No session_flag or duration column — session classification skipped')

In [ ]:
# ── 2.2  traffic_5G — aggregate to daily for time series ─────────────
traffic_5g_col = next(
    (c for c in df.columns if '5g' in c.lower() and 'traffic' in c.lower()), None
)

if traffic_5g_col:
    print(f'5G traffic column: {traffic_5g_col}')
    daily_5g = (
        df.groupby('date')[traffic_5g_col]
          .sum()
          .reset_index()
          .rename(columns={traffic_5g_col: 'traffic_5g', 'date': 'ds'})
    )
    daily_5g['ds'] = pd.to_datetime(daily_5g['ds'])
    daily_5g = daily_5g.sort_values('ds').reset_index(drop=True)
    print(f'Daily 5G series: {len(daily_5g)} days')
    print(daily_5g.describe())
else:
    daily_5g = None
    print('⚠️  No 5G traffic column found — 5G forecast section will be skipped')

In [ ]:
# ── 2.3  Brand traffic — aggregate by brand × day ────────────────────
brand_traffic_col = traffic_5g_col or next(
    (c for c in df.columns if 'traffic' in c.lower() and 'total' in c.lower()), None
) or next(
    (c for c in df.columns if c.startswith('dou_') or c == 'dou_total'), None
)

if 'brand' in df.columns and brand_traffic_col:
    top_brands = df['brand'].value_counts().head(5).index.tolist()
    df_top_brands = df[df['brand'].isin(top_brands)].copy()

    brand_daily = (
        df_top_brands
          .groupby(['date', 'brand'])[brand_traffic_col]
          .sum()
          .reset_index()
          .rename(columns={brand_traffic_col: 'traffic', 'date': 'ds'})
    )
    brand_daily['ds'] = pd.to_datetime(brand_daily['ds'])
    print(f'Brand daily table: {len(brand_daily)} rows, brands: {top_brands}')
    BRAND_TARGET = brand_traffic_col
else:
    brand_daily  = None
    BRAND_TARGET = None
    print('⚠️  Brand or traffic column missing — brand forecast skipped')

In [ ]:
# ── 2.4  Feature set for XGBoost (row-level) ─────────────────────────
# Numeric features only, exclude target-leaking columns
low_var_cols = ['mcc', 'mnc', 'roaming_direction', 'ccr_u', 'ccr_i_gx', 
                'ccr_u_gx', 'hour', 'reserved_field5']
EXCLUDE = {
    'session_flag', 'timestamp', 'date', 'msisdn', 'imsi',
    'home_cell', 'home_site', 'site_name', 'apn',
    'brand', 'model', 'tertype', 'generation', 'usertype',
    'mobility_class', 'user_class', 'ran_ne_vendor',
    'layer1name', 'layer2name', 'layer3name',
    'longitude', 'latitude',
}
num_cols = df.select_dtypes(include=[np.number]).columns
FEATURE_COLS = [c for c in num_cols if c not in EXCLUDE and c not in low_var_cols]
print(f'XGBoost feature set: {len(FEATURE_COLS)} columns')
print(FEATURE_COLS)

---
## 📊 Section 3 — Train / Test Split (Time-Aware)

In [ ]:
# ── 3.1  Chronological split: 80% train / 20% test ───────────────────
# Critical: we use time ordering, NOT random split, to avoid data leakage.
SPLIT_RATIO = 0.80
split_idx   = int(len(df) * SPLIT_RATIO)
split_date  = df['timestamp'].iloc[split_idx]

df_train = df.iloc[:split_idx].copy()
df_test  = df.iloc[split_idx:].copy()

print(f'Split date  : {split_date}')
print(f'Train rows  : {len(df_train):,}  ({len(df_train)/len(df)*100:.0f}%)')
print(f'Test rows   : {len(df_test):,}   ({len(df_test)/len(df)*100:.0f}%)')

# Same split for 5G daily series (if available)
if daily_5g is not None:
    n5g        = len(daily_5g)
    split5g    = int(n5g * SPLIT_RATIO)
    train_5g   = daily_5g.iloc[:split5g].copy()
    test_5g    = daily_5g.iloc[split5g:].copy()
    HORIZON    = len(test_5g)
    print(f'5G split    : train={len(train_5g)} days, test={len(test_5g)} days, horizon={HORIZON}')
else:
    HORIZON = 30   # default forecast horizon

---
## 🤖 Section A — Session Flag: XGBoost Classification
**Target:** `session_flag` (1 = active session, 0 = inactive)  
**Goal:** F1 > 0.72 · AUC > 0.85

In [ ]:
# ── A.1  Prepare X / y ───────────────────────────────────────────────
if SESSION_TARGET is None:
    print('Session target unavailable — skipping Section A')
else:
    feat_cols_session = [c for c in FEATURE_COLS
                         if c != SESSION_TARGET and df[c].dtype in [np.float64, np.int64]]

    X_train_s = df_train[feat_cols_session].fillna(0)
    y_train_s = df_train[SESSION_TARGET].astype(int)
    X_test_s  = df_test[feat_cols_session].fillna(0)
    y_test_s  = df_test[SESSION_TARGET].astype(int)

    print(f'Session classification features: {len(feat_cols_session)}')
    print(f'Train class balance: {y_train_s.value_counts().to_dict()}')
    print(f'Test  class balance: {y_test_s.value_counts().to_dict()}')

In [ ]:
# Vérifier les corrélations avec session_flag
corr = df[num_cols].corr()['session_flag'].abs().sort_values(ascending=False)
print("Top 15 corrélations avec session_flag :")
print(corr.head(15))

In [ ]:
print(f"Nombre de features: {len(FEATURE_COLS)}")
print("reserved_field5" in FEATURE_COLS)  # Doit être False

In [ ]:
# ── A.2  XGBoost classifier ──────────────────────────────────────────
if SESSION_TARGET is not None:
    # Handle class imbalance with scale_pos_weight
    neg = (y_train_s == 0).sum()
    pos = (y_train_s == 1).sum()
    scale_pos = neg / pos if pos > 0 else 1.0

    clf = xgb.XGBClassifier(
        n_estimators      = 300,
        max_depth         = 6,
        learning_rate     = 0.08,
        subsample         = 0.85,
        colsample_bytree  = 0.8,
        scale_pos_weight  = scale_pos,
        use_label_encoder = False,
        eval_metric       = 'logloss',
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = 0,
    )
    clf.fit(
        X_train_s, y_train_s,
        eval_set         = [(X_test_s, y_test_s)],
        verbose          = False,
    )

    # ── Evaluate ──────────────────────────────────────────────────────
    y_pred_s  = clf.predict(X_test_s)
    y_prob_s  = clf.predict_proba(X_test_s)[:, 1]

    acc_s   = accuracy_score(y_test_s, y_pred_s)
    f1_s    = f1_score(y_test_s, y_pred_s, average='binary', zero_division=0)
    prec_s  = precision_score(y_test_s, y_pred_s, zero_division=0)
    rec_s   = recall_score(y_test_s, y_pred_s, zero_division=0)
    auc_s   = roc_auc_score(y_test_s, y_prob_s) if len(np.unique(y_test_s)) > 1 else None

    print('\n=== XGBoost Session Flag — Classification Report ===')
    print(classification_report(y_test_s, y_pred_s, target_names=['Inactive','Active']))
    print(f'Accuracy  : {acc_s:.4f}')
    print(f'F1-Score  : {f1_s:.4f}   (target > 0.72) { "" if f1_s > 0.72 else "⚠️"}')
    print(f'Precision : {prec_s:.4f}')
    print(f'Recall    : {rec_s:.4f}')
    print(f'AUC-ROC   : {auc_s:.4f}  (target > 0.85) { "" if auc_s and auc_s > 0.85 else "⚠️"}')

In [ ]:
# ── A.3  Feature importance + confusion matrix ───────────────────────
if SESSION_TARGET is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('XGBoost Session Flag Classifier — Model Evaluation',
                 fontsize=14, fontweight='bold', color=HW['navy'])

    # Feature importance (top 15)
    ax = axes[0]
    fi = pd.Series(clf.feature_importances_, index=feat_cols_session)
    fi_top = fi.sort_values(ascending=True).tail(15)
    colors_fi = [HW['red'] if v == fi_top.max() else HW['blue'] for v in fi_top]
    ax.barh(fi_top.index, fi_top.values, color=colors_fi, alpha=0.88)
    ax.set_title('Top 15 Feature Importances', pad=12)
    ax.set_xlabel('Importance Score')
    ax.grid(axis='x', alpha=0.4)
    ax.grid(axis='y', visible=False)

    # Confusion matrix
    ax2 = axes[1]
    cm = confusion_matrix(y_test_s, y_pred_s)
    cmap_cm = mcolors.LinearSegmentedColormap.from_list('hw', ['#EFF6FF','#0093D5','#001F3F'])
    sns.heatmap(cm, ax=ax2, cmap=cmap_cm, annot=True, fmt='d',
                xticklabels=['Inactive','Active'],
                yticklabels=['Inactive','Active'],
                linewidths=2, linecolor='white')
    ax2.set_title('Confusion Matrix', pad=12)
    ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')
    # Metrics annotation
    ax2.text(1.05, 0.5,
             f'F1:  {f1_s:.3f}\nAUC: {auc_s:.3f}\nAcc: {acc_s:.3f}',
             transform=ax2.transAxes, fontsize=11, va='center',
             fontfamily='monospace', color=HW['navy'],
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#F0F9FF',
                       edgecolor=HW['blue'], linewidth=1.2))

    add_watermark(fig)
    plt.tight_layout()
    save_fig('figA1_session_flag_evaluation')
    plt.show()

> **Conclusion — Session Flag:**  
> XGBoost handles class imbalance via `scale_pos_weight`. F1 > 0.72 and AUC > 0.85 are the acceptance thresholds.  
> If targets are not met, try: (1) increasing `n_estimators`, (2) adding temporal lag features, (3) SMOTE oversampling.

---
## 📈 Section B — 5G Traffic: Prophet Time Series
**Target:** `traffic_5G` aggregated to daily  
**Model:** Facebook Prophet — handles trend + weekly seasonality

In [ ]:
# ── B.1  Prepare Prophet dataframe ──────────────────────────────────
if daily_5g is None:
    print('5G daily series unavailable — skipping Section B')
    prophet_mae   = None
    prophet_mape  = None
    prophet_fc    = None
else:
    # Prophet requires columns named 'ds' and 'y'
    train_p = train_5g.rename(columns={'traffic_5g': 'y'})[['ds','y']]
    test_p  = test_5g.rename(columns={'traffic_5g':  'y'})[['ds','y']]
    train_p['cap'] = train_p['y'].max() * 1.5
    test_p['cap'] = train_p['cap'].iloc[0]


    print(f'Prophet train: {len(train_p)} points')
    print(f'Prophet test : {len(test_p)}  points')
    print(f'y stats:\n{train_p["y"].describe().round(2)}')


In [ ]:
# ── B.2  Fit Prophet ─────────────────────────────────────────────────
def mape_capped(y_true, y_pred, min_threshold=8.0):
    """MAPE excluding near-zero days (log-value < 8 = raw < ~3000 bytes)."""
    mask = y_true > min_threshold
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) /
                           y_true[mask])) * 100

def smape(y_true, y_pred):
    """Symmetric MAPE — less sensitive to near-zero denominators."""
    return np.mean(2 * np.abs(y_pred - y_true) /
                   (np.abs(y_true) + np.abs(y_pred))) * 100

if daily_5g is not None:
    m_prophet = Prophet(
        seasonality_mode        = 'multiplicative',
        yearly_seasonality      = True,
        weekly_seasonality      = True,
        daily_seasonality       = False,
        changepoint_prior_scale = 0.05,
        seasonality_prior_scale = 5.0,
        interval_width          = 0.80,
    )
    m_prophet.fit(train_p)

    # Forecast over test period + 30 extra days
    future     = m_prophet.make_future_dataframe(periods=len(test_p) + 30, freq='D')
    forecast_p = m_prophet.predict(future)

    # Align predictions to test set dates
    pred_p = forecast_p[forecast_p['ds'].isin(test_p['ds'])]['yhat'].values
    true_p = test_p['y'].values

    # Clip negative predictions to 0
    pred_p = np.maximum(pred_p, 0)

    # Metrics
    prophet_mae    = mean_absolute_error(true_p, pred_p)
    prophet_rmse   = np.sqrt(mean_squared_error(true_p, pred_p))
    prophet_mape   = mape(true_p, pred_p)
    prophet_smape  = smape(true_p, pred_p)
    prophet_mape_f = mape_capped(true_p, pred_p)

    print(f'Prophet — MAE         : {prophet_mae:.2f}')
    print(f'Prophet — RMSE        : {prophet_rmse:.2f}')
    print(f'Prophet — MAPE        : {prophet_mape:.2f}%   (target <20%)')
    print(f'Prophet — sMAPE       : {prophet_smape:.2f}%')
    print(f'Prophet — MAPE filtered: {prophet_mape_f:.2f}%  (excl. near-zero days)')

    # Future 30-day forecast
    prophet_fc = forecast_p.tail(30)[['ds','yhat','yhat_lower','yhat_upper']].copy()
    prophet_fc['yhat'] = prophet_fc['yhat'].clip(lower=0)

In [ ]:
# ── B.3  Prophet visualization ───────────────────────────────────────
if daily_5g is not None:
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    fig.suptitle('Prophet — 5G Traffic Forecast', fontsize=14,
                 fontweight='bold', color=HW['navy'])

    # Full history + forecast
    ax = axes[0]
    ax.plot(train_p['ds'], train_p['y'],
            color=HW['blue'], linewidth=1.5, label='Historical (train)', alpha=0.85)
    ax.plot(test_p['ds'],  test_p['y'],
            color=HW['navy'], linewidth=1.5, label='Actual (test)', alpha=0.85)
    aligned = forecast_p[forecast_p['ds'].isin(test_p['ds'])]
    ax.plot(aligned['ds'], aligned['yhat'],
            color=HW['red'], linewidth=2, linestyle='--', label='Prophet forecast')
    ax.fill_between(aligned['ds'],
                    aligned['yhat_lower'].clip(lower=0),
                    aligned['yhat_upper'],
                    alpha=0.15, color=HW['red'], label='80% CI')
    ax.set_title('5G Traffic — Historical vs Prophet Forecast', pad=12)
    ax.set_ylabel('Daily Traffic Volume')
    ax.legend()
    ax.text(0.02, 0.97,
            f'MAE: {prophet_mae:.2f}  |  MAPE: {prophet_mape:.1f}%',
            transform=ax.transAxes, fontsize=10, va='top',
            color=HW['navy'], fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='#EFF6FF',
                      edgecolor=HW['blue'], alpha=0.9))

    # Components
    ax2 = axes[1]
    ax2.plot(prophet_fc['ds'], prophet_fc['yhat'],
             color=HW['red'], linewidth=2.5, label='30-day forecast')
    ax2.fill_between(prophet_fc['ds'],
                     prophet_fc['yhat_lower'].clip(lower=0),
                     prophet_fc['yhat_upper'],
                     alpha=0.18, color=HW['red'])
    ax2.set_title('30-Day Ahead 5G Traffic Projection', pad=12)
    ax2.set_ylabel('Projected Traffic')
    ax2.legend()

    add_watermark(fig)
    plt.tight_layout()
    save_fig('figB1_prophet_5g_forecast')
    plt.show()

---
## 📈 Section C — 5G Traffic: ARIMA Baseline
**Model:** pmdarima auto_arima — automatic order selection  
**Purpose:** Statistical baseline to compare with Prophet and XGBoost

In [ ]:
import numpy as np
from pmdarima import auto_arima

# Log-transform pour stabiliser la variance
train_log = np.log1p(train_p['y'])

# Forcer d >= 1
model_arima = auto_arima(
    train_log,
    start_p=1, max_p=5,
    start_q=1, max_q=5,
    d=1,                    # Forcer différenciation
    seasonal=False,
    trace=False,
    stepwise=True,
    error_action='ignore',
    suppress_warnings=True
)

print(f"Best ARIMA: {model_arima.order}")

# Prédictions
pred_log = model_arima.predict(n_periods=len(test_p))
pred = np.expm1(pred_log)  # Inverse log1p
rmse = np.sqrt(np.mean((test_p['y'].values - pred)**2))
mape = np.mean(np.abs((test_p['y'].values - pred) / test_p['y'].values)) * 100

print(f"RMSE: {rmse:,.2f}")
print(f"MAPE: {mape:.2f}%")
# Métriques
mae = np.mean(np.abs(test_p['y'].values - pred))
print(f"MAE: {mae:,.2f}")

In [ ]:
# ── C.2  ARIMA diagnostic plot ───────────────────────────────────────
if daily_5g is not None and PMDARIMA_OK:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(test_5g['ds'].values,    y_test_arima, color=HW['blue'],
            linewidth=1.5, label='Actual')
    ax.plot(test_5g['ds'].values,    pred_arima,   color=HW['red'],
            linewidth=2, linestyle='--', label=f'ARIMA  (MAE={arima_mae:.2f})')
    ax.set_title('ARIMA Forecast — 5G Traffic', pad=14)
    ax.set_ylabel('Daily 5G Traffic')
    ax.legend()
    ax.text(0.02, 0.97,
            f'Order {m_arima.order}  |  MAE {arima_mae:.2f}  |  MAPE {arima_mape:.1f}%',
            transform=ax.transAxes, fontsize=10, va='top', color=HW['navy'],
            bbox=dict(boxstyle='round', facecolor='#FFF7ED', edgecolor=HW['amber']))
    add_watermark(fig)
    save_fig('figC1_arima_5g_forecast')
    plt.show()

---
## 📈 Section D — 5G Traffic: XGBoost Regression
**Model:** XGBoost with lag features  
**Advantage:** Captures non-linear relationships between network KPIs and traffic

In [ ]:
# ── D.1  Build lag features for XGBoost regression ──────────────────
if daily_5g is None:
    print('5G data unavailable — skipping Section D')
    xgb_reg_mae  = None
    xgb_reg_mape = None
else:
    ts_feat = daily_5g.copy().rename(columns={'traffic_5g': 'y'})

    # Lag features: 1, 2, 3, 7, 14 days
    for lag in [1, 2, 3, 7, 14]:
        ts_feat[f'lag_{lag}'] = ts_feat['y'].shift(lag)

    # Rolling statistics
    ts_feat['roll7_mean'] = ts_feat['y'].shift(1).rolling(7).mean()
    ts_feat['roll7_std']  = ts_feat['y'].shift(1).rolling(7).std()

    # Calendar features
    ts_feat['dow']     = ts_feat['ds'].dt.dayofweek
    ts_feat['month']   = ts_feat['ds'].dt.month
    ts_feat['week_num']= ts_feat['ds'].dt.isocalendar().week.astype(int)

    ts_feat = ts_feat.dropna().reset_index(drop=True)

    LAG_FEATURES = [c for c in ts_feat.columns if c not in ['ds', 'y']]
    split_ts = int(len(ts_feat) * SPLIT_RATIO)

    X_tr_r = ts_feat[LAG_FEATURES].iloc[:split_ts]
    y_tr_r = ts_feat['y'].iloc[:split_ts]
    X_te_r = ts_feat[LAG_FEATURES].iloc[split_ts:]
    y_te_r = ts_feat['y'].iloc[split_ts:]

    print(f'XGB regression features: {LAG_FEATURES}')
    print(f'Train: {len(X_tr_r)}, Test: {len(X_te_r)}')

In [ ]:
# ── D.2  XGBoost regressor ───────────────────────────────────────────
mask = y_te_r.values != 0

if daily_5g is not None:
    reg = xgb.XGBRegressor(
        n_estimators     = 400,
        max_depth        = 5,
        learning_rate    = 0.06,
        subsample        = 0.85,
        colsample_bytree = 0.8,
        reg_alpha        = 0.1,
        reg_lambda       = 1.0,
        random_state     = 42,
        n_jobs           = -1,
        verbosity        = 0,
    )
    reg.fit(X_tr_r, y_tr_r,
            eval_set        = [(X_te_r, y_te_r)],
            verbose         = False)

    pred_xgb_r    = np.maximum(reg.predict(X_te_r), 0)
    xgb_reg_mae   = mean_absolute_error(y_te_r, pred_xgb_r)
    xgb_reg_rmse  = np.sqrt(mean_squared_error(y_te_r, pred_xgb_r))
    xgb_reg_r2    = r2_score(y_te_r, pred_xgb_r)
    xgb_reg_mape = np.mean(np.abs((y_te_r.values[mask] - pred_xgb_r[mask]) / y_te_r.values[mask])) * 100


    print(f'XGBoost Regression — MAE:  {xgb_reg_mae:.2f}')
    print(f'XGBoost Regression — RMSE: {xgb_reg_rmse:.2f}')
    print(f'XGBoost Regression — R²:   {xgb_reg_r2:.4f}')
    print(f'XGBoost Regression — MAPE: {xgb_reg_mape:.2f}%')

In [ ]:
# ── D.3  XGBoost regression plot ────────────────────────────────────
if daily_5g is not None:
    dates_te = ts_feat['ds'].iloc[split_ts:].values

    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    fig.suptitle('XGBoost Regression — 5G Traffic Forecast',
                 fontsize=14, fontweight='bold', color=HW['navy'])

    ax = axes[0]
    ax.plot(dates_te, y_te_r.values,  color=HW['blue'],  lw=1.5, label='Actual')
    ax.plot(dates_te, pred_xgb_r,     color=HW['red'],   lw=2.0, ls='--', label=f'XGBoost (MAE={xgb_reg_mae:.2f})')
    ax.set_title('Actual vs XGBoost Predicted — Test Period', pad=12)
    ax.set_ylabel('Daily 5G Traffic')
    ax.legend()
    ax.text(0.02, 0.97,
            f'R²={xgb_reg_r2:.3f}  |  MAPE={xgb_reg_mape:.1f}%',
            transform=ax.transAxes, fontsize=10, va='top', color=HW['navy'],
            bbox=dict(boxstyle='round', facecolor='#F0FFF4', edgecolor=HW['green']))

    # Feature importance
    ax2 = axes[1]
    fi_reg = pd.Series(reg.feature_importances_, index=LAG_FEATURES).sort_values(ascending=True)
    bar_c  = [HW['red'] if v == fi_reg.max() else HW['blue'] for v in fi_reg]
    ax2.barh(fi_reg.index, fi_reg.values, color=bar_c, alpha=0.88)
    ax2.set_title('XGBoost Feature Importance (Regression)', pad=12)
    ax2.set_xlabel('Importance')
    ax2.grid(axis='x', alpha=0.4)
    ax2.grid(axis='y', visible=False)

    add_watermark(fig)
    plt.tight_layout()
    save_fig('figD1_xgboost_5g_regression')
    plt.show()

---
## 📊 Section E — Brand Traffic Forecast
**Approach:** Per-brand Prophet + XGBoost, select winner by MAE  
**Brands:** Top 5 by data volume

In [ ]:
# ── E.1  Brand Prophet loop ──────────────────────────────────────────
brand_scores = []
brand_forecasts_list = []

if brand_daily is None:
    print('Brand data unavailable — skipping Section E')
else:
    brands = brand_daily['brand'].unique()
    print(f'Forecasting {len(brands)} brands: {brands.tolist()}')

    for brand in brands:
        bdf = brand_daily[brand_daily['brand'] == brand].copy()
        bdf = bdf.sort_values('ds').reset_index(drop=True)

        if len(bdf) < 14:   # need at least 2 weeks
            print(f'  {brand}: too few points ({len(bdf)}) — skipped')
            continue

        nb_split = int(len(bdf) * SPLIT_RATIO)
        b_train  = bdf.iloc[:nb_split].rename(columns={'traffic': 'y'})[['ds','y']]
        b_test   = bdf.iloc[nb_split:].rename(columns={'traffic': 'y'})[['ds','y']]

        # Prophet
        try:
            mp = Prophet(seasonality_mode='multiplicative',
                         weekly_seasonality=True, yearly_seasonality=False,
                         changepoint_prior_scale=0.1, interval_width=0.80)
            mp.fit(b_train)
            future_b = mp.make_future_dataframe(periods=len(b_test)+30, freq='D')
            fc_b     = mp.predict(future_b)
            preds_b  = fc_b[fc_b['ds'].isin(b_test['ds'])]['yhat'].clip(lower=0).values

            if len(preds_b) == len(b_test):
                b_mae  = mean_absolute_error(b_test['y'].values, preds_b)
                b_mape = mape(b_test['y'].values, preds_b)

                # 30-day future forecast
                future_30 = fc_b.tail(30)[['ds','yhat']].copy()
                future_30['brand']  = brand
                future_30['target'] = BRAND_TARGET or 'traffic'
                future_30['model']  = 'prophet'
                future_30.rename(columns={'yhat': 'forecast'}, inplace=True)
                brand_forecasts_list.append(future_30)

                brand_scores.append({
                    'brand': brand, 'model': 'prophet',
                    'mae': round(b_mae,3), 'mape': round(b_mape,2),
                    'n_test': len(b_test),
                })
                print(f'  {brand:<20s} Prophet  MAE={b_mae:.2f}  MAPE={b_mape:.1f}%')
        except Exception as e:
            print(f'  {brand}: Prophet failed ({e})')

In [ ]:
# ── E.2  Brand forecast visualization ────────────────────────────────
from IPython.display import display
if brand_daily is not None and len(brand_forecasts_list) > 0:
    all_fc_brands = pd.concat(brand_forecasts_list, ignore_index=True)

    fig = px.line(
        all_fc_brands,
        x     = 'ds',
        y     = 'forecast',
        color = 'brand',
        title = '30-Day Traffic Forecast by Device Brand',
        labels= {'ds': 'Date', 'forecast': 'Forecast Traffic', 'brand': 'Brand'},
        color_discrete_sequence=PALETTE,
    )
    fig.update_layout(**PLOTLY_LAYOUT)
    fig.write_html(str(FIG_DIR / 'figE1_brand_forecast.html'))
    display(fig)

    if brand_scores:
        bs_df = pd.DataFrame(brand_scores).sort_values('mae')
        print('\nBrand Forecast Scores:')
        print(bs_df.to_string(index=False))

        # MAE comparison bar
        fig2, ax = plt.subplots(figsize=(12, 4))
        colors_b = [HW['red'] if row.mae == bs_df['mae'].min() else HW['blue']
                    for row in bs_df.itertuples()]
        ax.bar(bs_df['brand'], bs_df['mae'], color=colors_b, alpha=0.88)
        ax.set_title('Prophet MAE by Brand', pad=12)
        ax.set_ylabel('MAE (traffic units)')
        for i, row in enumerate(bs_df.itertuples()):
            ax.text(i, row.mae + bs_df['mae'].max()*0.01, f'{row.mae:.1f}',
                    ha='center', fontsize=8.5, color=HW['navy'], fontweight='bold')
        add_watermark(fig2)
        save_fig('figE2_brand_mae_comparison')
        display(fig2)

---
## 📊 Section 9 — Model Comparison Dashboard
Side-by-side summary of all models with Huawei-styled visuals.

In [ ]:
# ── 9.1  Compile comparison table ────────────────────────────────────
rows = []

if SESSION_TARGET is not None:
    rows.append({'model':'XGBoost','target':'session_flag','metric':'F1',
                 'value':round(f1_s,4),'pass':f1_s>0.72})
    if auc_s:
        rows.append({'model':'XGBoost','target':'session_flag','metric':'AUC-ROC',
                     'value':round(auc_s,4),'pass':auc_s>0.85})

if daily_5g is not None:
    if prophet_mae is not None:
        rows.append({'model':'Prophet','target':'traffic_5G','metric':'MAE',
                     'value':round(prophet_mae,2),'pass':True})
        rows.append({'model':'Prophet','target':'traffic_5G','metric':'MAPE',
                     'value':round(prophet_mape,2),'pass':prophet_mape<20})
    if 'arima_mae' in dir() and arima_mae is not None:
        rows.append({'model':'ARIMA','target':'traffic_5G','metric':'MAE',
                     'value':round(arima_mae,2),'pass':True})
        rows.append({'model':'ARIMA','target':'traffic_5G','metric':'MAPE',
                     'value':round(arima_mape,2),'pass':arima_mape<20})
    if xgb_reg_mae is not None:
        rows.append({'model':'XGBoost','target':'traffic_5G','metric':'MAE',
                     'value':round(xgb_reg_mae,2),'pass':True})
        rows.append({'model':'XGBoost','target':'traffic_5G','metric':'MAPE',
                     'value':round(xgb_reg_mape,2),'pass':xgb_reg_mape<20})
        rows.append({'model':'XGBoost','target':'traffic_5G','metric':'R²',
                     'value':round(xgb_reg_r2,4),'pass':xgb_reg_r2>0.7})

compare_df = pd.DataFrame(rows)
if len(compare_df) > 0:
    print('=== MODEL COMPARISON ===')
    print(compare_df.to_string(index=False))
    print(f'\nPassed all targets: {compare_df["pass"].all()}')

In [ ]:
# ── 9.2  Model comparison figure ─────────────────────────────────────
if len(compare_df) > 0 and daily_5g is not None:
    # MAE comparison for 5G target
    mae_rows = compare_df[(compare_df['target']=='traffic_5G') &
                          (compare_df['metric']=='MAE')].copy()

    if len(mae_rows) > 0:
        best_model = mae_rows.loc[mae_rows['value'].idxmin(), 'model']
        fig, ax = plt.subplots(figsize=(10, 4))
        colors_m = [HW['red'] if m == best_model else HW['blue']
                    for m in mae_rows['model']]
        bars = ax.bar(mae_rows['model'], mae_rows['value'],
                      color=colors_m, alpha=0.88, zorder=3)
        for bar, v in zip(bars, mae_rows['value']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5,
                    f'{v:.2f}', ha='center', fontsize=10,
                    fontweight='bold', color=HW['navy'])
        ax.set_title('Model Comparison — MAE on 5G Traffic Forecast\n'
                     f'Best: {best_model} (highlighted in red)', pad=14)
        ax.set_ylabel('Mean Absolute Error')
        legend_patches = [
            mpatches.Patch(color=HW['red'],  label=f'Best: {best_model}'),
            mpatches.Patch(color=HW['blue'], label='Other models'),
        ]
        ax.legend(handles=legend_patches)
        add_watermark(fig)
        save_fig('fig09_model_comparison_mae')
        plt.show()
        print(f'\n🏆 Best model for 5G traffic: {best_model}')

---
## 💾 Section 10 — Save Artifacts

In [ ]:
# ── 10.1  prediction_scores.parquet ──────────────────────────────────
# Format: one row per (region/brand) with mae/mape per model
# Used by: /api/forecast/scores → Forecasting.jsx model comparison table

score_rows = []

# 5G target scores (global — single region for now)
base_row = {'region': 'ALL', 'target': 'traffic_5G'}

if prophet_mae is not None:
    base_row.update({'prophet_mae': prophet_mae, 'prophet_mape': prophet_mape})
if 'arima_mae' in dir() and arima_mae is not None:
    base_row.update({'arima_mae': arima_mae, 'arima_mape': arima_mape})
if xgb_reg_mae is not None:
    base_row.update({'xgboost_mae': xgb_reg_mae, 'xgboost_mape': xgb_reg_mape,
                     'xgboost_r2':  xgb_reg_r2})

# Determine winner for this row
mae_by_model = {k.replace('_mae',''): v for k, v in base_row.items() if k.endswith('_mae') and k!='region'}
if mae_by_model:
    base_row['winner'] = min(mae_by_model, key=mae_by_model.get)

score_rows.append(base_row)

# Per-brand rows (from Section E)
if brand_scores:
    for bs in brand_scores:
        score_rows.append({
            'region'      : bs['brand'],
            'target'      : BRAND_TARGET or 'traffic',
            'prophet_mae' : bs['mae'],
            'prophet_mape': bs['mape'],
            'winner'      : 'prophet',
        })

scores_df = pd.DataFrame(score_rows)
scores_path = MODEL_DIR / 'prediction_scores.parquet'
scores_df.to_parquet(scores_path, index=False)
print(f'✅ Saved: {scores_path}')
print(scores_df.to_string(index=False))

In [ ]:
# ── 10.2  forecasts.parquet ──────────────────────────────────────────
# Format: {date, region, forecast, model, target}
# Used by: /api/forecast/list → Forecasting.jsx chart

fc_rows = []

# Prophet 30-day forecast for 5G
if daily_5g is not None and prophet_fc is not None:
    for _, row in prophet_fc.iterrows():
        fc_rows.append({
            'date'    : str(row['ds'].date()),
            'region'  : 'ALL',
            'forecast': round(float(row['yhat']),2),
            'model'   : 'prophet',
            'target'  : 'traffic_5G',
        })

# XGBoost 5G regression predictions on test set (for history chart)
if daily_5g is not None and xgb_reg_mae is not None:
    dates_xgb = ts_feat['ds'].iloc[split_ts:].values
    for d, p in zip(dates_xgb, pred_xgb_r):
        fc_rows.append({
            'date'    : str(pd.Timestamp(d).date()),
            'region'  : 'ALL',
            'forecast': round(float(p), 2),
            'model'   : 'xgboost',
            'target'  : 'traffic_5G',
        })

# Brand forecasts
if brand_daily is not None and len(brand_forecasts_list) > 0:
    for _, row in all_fc_brands.iterrows():
        fc_rows.append({
            'date'    : str(pd.Timestamp(row['ds']).date()),
            'region'  : row['brand'],
            'forecast': round(float(row['forecast']),2),
            'model'   : 'prophet',
            'target'  : BRAND_TARGET or 'traffic',
        })

if fc_rows:
    fc_df = pd.DataFrame(fc_rows)
    fc_path = MODEL_DIR / 'forecasts.parquet'
    fc_df.to_parquet(fc_path, index=False)
    print(f'✅ Saved: {fc_path}  ({len(fc_df):,} rows)')
    print(fc_df.head())
else:
    print('No forecast rows to save')

In [ ]:
# ── 10.3  forecast_results.json ──────────────────────────────────────
# High-level summary for /api/forecast/preview (dashboard tiles)

# Compute 5G growth: first vs last forecast point
fc_5g_growth = None
fc_5g_trend  = 'stable'
if prophet_fc is not None:
    first_fc = float(prophet_fc['yhat'].iloc[0])
    last_fc  = float(prophet_fc['yhat'].iloc[-1])
    if first_fc > 0:
        fc_5g_growth = round(((last_fc - first_fc) / first_fc) * 100, 1)
        fc_5g_trend  = 'up' if fc_5g_growth > 2 else 'down' if fc_5g_growth < -2 else 'stable'

# Session active pct (from XGBoost classifier)
session_active_pct = None
if SESSION_TARGET is not None:
    # Use the predicted probability mean on test set as proxy
    session_active_pct = round(float(y_prob_s.mean()) * 100, 1)

# Top brand by total forecasted traffic
top_brand = None
if brand_scores:
    top_brand = min(brand_scores, key=lambda x: x['mae'])['brand']

best_xgb_mae = min(
    [v for v in [xgb_reg_mae] if v is not None],
    default=None
)

forecast_results = {
    'generated_at'          : datetime.now().isoformat(),
    'traffic_5g_growth_pct' : fc_5g_growth,
    'traffic_5g_trend'      : fc_5g_trend,
    'session_active_pct'    : session_active_pct,
    'session_trend'         : 'stable',
    'top_brand'             : top_brand,
    'xgb_best_mae'          : round(best_xgb_mae, 3) if best_xgb_mae else None,
    'models_run'            : {
        'prophet'  : daily_5g is not None and prophet_mae is not None,
        'arima'    : daily_5g is not None and PMDARIMA_OK and (arima_mae is not None if 'arima_mae' in dir() else False),
        'xgboost'  : xgb_reg_mae is not None,
        'session'  : SESSION_TARGET is not None,
    },
    'prophet_5g_mae'   : round(prophet_mae,  3) if prophet_mae  is not None else None,
    'arima_5g_mae'     : round(arima_mae,    3) if ('arima_mae' in dir() and arima_mae is not None) else None,
    'xgboost_5g_mae'   : round(xgb_reg_mae,  3) if xgb_reg_mae is not None else None,
    'session_f1'       : round(f1_s,         3) if SESSION_TARGET is not None else None,
    'session_auc'      : round(auc_s,         3) if (SESSION_TARGET is not None and auc_s) else None,
}

json_path = OUT_DIR / 'forecast_results.json'
with open(json_path, 'w') as f:
    json.dump(forecast_results, f, indent=2, default=str)
print(f'✅ Saved: {json_path}')
print(json.dumps(forecast_results, indent=2, default=str))

In [ ]:
# ── 10.4  Final summary ───────────────────────────────────────────────
figs = sorted(FIG_DIR.glob('fig[ABCDE09]*.png')) + sorted(FIG_DIR.glob('fig[ABCDE09]*.html'))

print('\n' + '='*60)
print('NOTEBOOK 02 — KPI FORECASTING COMPLETE')
print('='*60)
print(f'  {"Model":<12s}  {"Target":<16s}  {"Key Metric":<25s}  {"Value"}')
print(f'  {"-"*12}  {"-"*16}  {"-"*25}  {"-"*10}')

rows = [
    ('XGBoost', 'session_flag', 'F1 (target >0.72)',  f'{f1_s:.4f}',    '✅' if f1_s  > 0.72 else '⚠️'),
    ('XGBoost', 'session_flag', 'AUC (target >0.85)', f'{auc_s:.4f}',   '✅' if auc_s > 0.85 else '⚠️'),
    ('Prophet', 'traffic_5G',   'MAE',                f'{prophet_mae:.2f}',  ''),
    ('Prophet', 'traffic_5G',   'MAPE (target <20%)', f'{prophet_mape:.1f}%','⚠️' if prophet_mape > 20 else '✅'),
    ('ARIMA',   'traffic_5G',   'MAE',                f'{arima_mae:.2f}',    ''),
    ('ARIMA',   'traffic_5G',   'MAPE (target <20%)', f'{arima_mape:.1f}%',  '✅' if arima_mape < 20 else '⚠️'),
    ('XGBoost', 'traffic_5G',   'MAE',                f'{xgb_reg_mae:.2f}',  ''),
    ('XGBoost', 'traffic_5G',   'MAPE (target <20%)', f'{xgb_reg_mape:.1f}%','✅' if xgb_reg_mape < 20 else '⚠️'),
    ('XGBoost', 'traffic_5G',   'R² (target >0.70)',  f'{xgb_reg_r2:.4f}',  '✅' if xgb_reg_r2 > 0.70 else '⚠️'),
]
for model, target, metric, value, status in rows:
    print(f'  {model:<12s}  {target:<16s}  {metric:<25s}  {value}  {status}')
print(f'\nArtifacts saved:')
print(f'  {MODEL_DIR}/prediction_scores.parquet')
print(f'  {MODEL_DIR}/forecasts.parquet')
print(f'  {OUT_DIR}/forecast_results.json')
print(f'  {len(figs)} figures in {FIG_DIR}')

print('\nNext → Run 03_Churn_EDA.ipynb')

---
## Summary

| Section | Model | Target | Key Metric | Acceptance |
|---------|-------|--------|------------|------------|
| A | XGBoost Classifier | `session_flag` | F1, AUC-ROC | F1 > 0.72, AUC > 0.85 |
| B | Prophet | `traffic_5G` | MAE, MAPE | MAPE < 20% |
| C | ARIMA (auto) | `traffic_5G` | MAE, MAPE | MAPE < 20% |
| D | XGBoost Regressor | `traffic_5G` | MAE, R² | R² > 0.70 |
| E | Prophet per brand | Brand traffic | MAE | — |

All outputs are in `models/prediction/` and `data/outputs/` — directly consumed by the FastAPI endpoints in `analytics_api.py`.  
The NOC Dashboard's Forecasting page and Overview forecast tiles will be populated immediately after running this notebook.

---
*SpiriCom · NOC Intelligence Platform · Huawei Technologies Tunisia · PFE 2026*